# Cellbender

## shell 

In [ ]:
cd RNA_out_tar

In [ ]:
my_func() {
  md5sum -c $1
}
export -f my_func
ls *.md5 | parallel -j 16 my_func

In [ ]:
#run in shell
for sample in `ls *.tar`
do
mkdir /data2/liyanguo/RNA_out/$(basename $sample .tar)
tar -xvf $sample -C /data2/liyanguo/RNA_out/$(basename $sample .tar)
done

In [ ]:
ls /data2/liyanguo/RNA_out_tar/*.tar > sample.list
split -l 50 sample.list

## Convert to h5ad format

In [1]:
import scanpy as sc
import numpy as np 
import pandas as pd 
import anndata as ad
from glob import glob
from tqdm import tqdm
import os
from multiprocessing import Pool

In [2]:
files = glob('/data2/liyanguo/RNA_out/*')

In [ ]:
len(files)

In [4]:
def to_h5ad(i):
    if os.path.exists(f'{i}/GeneFull_Ex50pAS/raw'):
        adata = sc.read_10x_mtx(f'{i}/GeneFull_Ex50pAS/raw')
        sample_name = i.split('/')[-1]
        adata.write_h5ad(f"{i}/{sample_name}.h5ad")

In [6]:
with Pool(16) as p:
     p.map(to_h5ad, files)

In [ ]:
# Run in shell, remove the redundant files generated by cellbender
#rm -rf RNA_out/*/GeneFull_Ex50pAS
#rm RNA_out/*/*.h5
#rm RNA_out/*/ckpt.tar.gz
#rm RNA_out/*/*.h5ad
#rm RNA_out_tar/*

## Cellbender for scRNA-seq generated by celescope and STARsolo with parameter GeneFull_Ex50pAS, which is introns: preioritize >50% overlap with exons.

In [ ]:
#!/bin/bash
#shell script in cellbender.sh
sample_list=$1
for sample in $(</data2/liyanguo/$sample_list);
do
    export CUDA_VISIBLE_DEVICES=$2;
    cd /data2/liyanguo/RNA_out/$sample/;
    input=$sample".h5ad";
    output=$sample"_rb.h5";
    cellbender remove-background \
    --input  $input \
    --output $output \
    --expected-cells 20000 \
    --total-droplets-included 60000 \
    --learning-rate 0.000001 \
    --cuda \
    --checkpoint-mins 90 \
    --estimator-multiple-cpu \
    --force-empty-umi-prior 200 \
    --projected-ambient-count-threshold 1 \
    --posterior-batch-size 1024 --epochs 100 --fpr 0.01 --low-count-threshold 5;
    rm *.h5
done

In [ ]:
# V100 GPU NUM
sh cellbender.sh sample.list 5 &

In [ ]:
for sample in ;
do
    export CUDA_VISIBLE_DEVICES=0;
    cd /data2/liyanguo/RNA_out/$sample/;
    input=$sample".h5ad";
    output=$sample"_rb.h5";
    cellbender remove-background \
    --input  $input \
    --output $output \
    --expected-cells 20000 \
    --total-droplets-included 60000 \
    --learning-rate 0.000001 \
    --cuda \
    --checkpoint-mins 90 \
    --estimator-multiple-cpu \
    --force-empty-umi-prior 200 \
    --projected-ambient-count-threshold 1 \
    --posterior-batch-size 1024 --epochs 100 --fpr 0.01 --low-count-threshold 5;
    rm *.h5
done &

In [ ]:
# If we can not get reasonable result, remove parameter --force-empty-umi-prior 200

## Copy result of Cellbender for VDJ

In [ ]:
#run in shell
cd /localdisk/immune/0_raw_immune1k/celescope_v2/MyImmuCell_scRNA

for sample in `ls *.tar`
do
mkdir -p /home/liyanguo/MyImmuCell/01_rawdata/MyImmuCell_scRNA/$(basename $sample .tar)/outs/filtered/
cp /localdisk/immune/0_raw_immune1k/celescope_v2/cellbender/$(basename $sample .tar)/*_rb_cell_barcodes.csv \
    /home/liyanguo/MyImmuCell/01_rawdata/MyImmuCell_scRNA/$(basename $sample .tar)/outs/filtered/barcodes.tsv
gzip /home/liyanguo/MyImmuCell/01_rawdata/MyImmuCell_scRNA/$(basename $sample .tar)/outs/filtered/barcodes.tsv
done